# Train an SO101-Nexus policy with demo-seeded PPO on MuJoCo Warp

Behavior-cloning (BC) seeded PPO on the GPU-parallel **MuJoCo Warp** backend: the same CleanRL PPO recipe as `ppo_warp.py`, plus BC pretraining and a persistent BC loss from 10 teleop demonstrations. This targets PPO's one known weakness on `WarpPickLift-v1`: seed 5 gets stuck at a grasp-hold-at-table local optimum under vanilla PPO (`best_success=0.037`). Same 30M-step recipe, demo-seeding alone rescues it to `best_success=0.993`, `final_success=0.983`.

## 1. Check the GPU

Runtime -> Change runtime type -> GPU.

In [ ]:
!nvidia-smi -L

## 2. Install

The example scripts live in `examples/`, not the wheel, so clone the repo and install the `warp` + `train` extras.


If you hit `ModuleNotFoundError: No module named 'so101_nexus._reproducibility'` (or any `so101_nexus` submodule) in a later cell, it is not a stale wheel: `pip install -e` adds the package to `sys.path` via a `.pth` file that Python only reads at interpreter startup. This kernel was already running before the install cell executed, so `!python ...` cells (fresh subprocesses) pick up the new install but in-kernel `import so101_nexus...` cells do not. The install cell below also inserts `src/` onto `sys.path` directly so both paths work without a runtime restart.

In [ ]:
!git clone --depth 1 https://github.com/johnsutor/so101-nexus.git
%cd so101-nexus
!pip uninstall -y so101-nexus 2>/dev/null || true
!pip install -q -e ".[warp,train]" "imageio[ffmpeg]"
# `pip install -e` registers the package via a .pth file that Python only
# reads when an interpreter starts. This kernel was already running before
# the install above, so `!python ...` cells (fresh subprocesses) see the
# package but in-kernel `import so101_nexus...` cells below will not unless
# `src/` is also put on sys.path explicitly, here.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

## 3. Choose the seed

`WarpPickLift-v1` is the only environment this recipe is tuned for -- the demo dataset ([`johnsutor/MuJoCoPickLift`](https://huggingface.co/datasets/johnsutor/MuJoCoPickLift)) is pick-lift-specific. Try seed 5 to reproduce the rescued-seed result, or any other seed to sanity-check that demo-seeding does not regress an already-solved one.

In [ ]:
SEED = 5
STEPS = 30_000_000
print(f"Training WarpPickLift-v1 for {STEPS:,} env steps, seed={SEED}")

## 4. Launch TensorBoard

Run before training so the dashboard picks up the new run. Watch `charts/success_rate` and `losses/bc_loss` (the persistent demo-anchoring term) alongside the usual PPO charts.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 5. Train

1024 GPU-parallel worlds, fixed-horizon episodes, entropy annealed 0.03 -> 0.005, staggered resets, obs/reward normalization, and a CleanRL-style optimizer budget -- identical to `ppo_warp.py`. On top of that: the actor is BC-pretrained on the 10 demo episodes (`--bc-pretrain-updates 2000`) before online PPO starts, and a persistent BC loss (`--bc-coef 0.1`) keeps anchoring it toward demo actions throughout training. `--use-demos false` recovers `ppo_warp.py` exactly. `best_agent.pt` is saved on every success-rate improvement. Reduce `--num-envs` on smaller GPUs.

MuJoCo eval rendering is off on Colab (`--eval-freq 0`); step 6 evaluates on the Warp backend instead.

In [ ]:
!python examples/bc_ppo_warp.py --exp-name colab --seed {SEED} \
    --total-timesteps {STEPS} --eval-freq 0

## 6. Evaluate the trained policy

Deterministic eval (no exploration noise) across 512 Warp worlds. Reports `success_rate(ever)`, hold rate at the final step, and mean return. `eval_warp.py` is checkpoint-format agnostic (it matches state-dict keys, not the training script that produced them), so it works on `bc_ppo_warp.py` checkpoints unchanged.

In [ ]:
CKPT = "runs/WarpPickLift-v1__colab__*/best_agent.pt"
!python examples/eval_warp.py --env-id WarpPickLift-v1 --checkpoint "{CKPT}"

## 7. Watch a sample rollout

The Warp backend runs 1024 worlds in parallel and does not render, so you cannot watch a single agent from training. This renders one deterministic episode of the trained policy in the matching MuJoCo backend (same policy weights and observation normalization, with a slight physics gap versus Warp). The mp4 plays inline below.


In [ ]:
# Render one deterministic rollout of the trained policy and play it inline.
import glob

from IPython.display import Video

from examples.bc_ppo_warp import rollout_video_from_checkpoint

ENV_ID = "WarpPickLift-v1"
checkpoint = sorted(glob.glob(CKPT))[-1]
print(f"Rendering sample rollout from {checkpoint}")

metrics, video_path = rollout_video_from_checkpoint(
    checkpoint,
    ENV_ID,
    control_mode="pd_joint_delta_pos",
    episode_length=512,
    hidden_dim=256,
    seed=12345,
)
print(
    f"rollout success_rate={metrics['eval/success_rate']:.2f} "
    f"return={metrics['eval/return']:.2f} "
    f"ep_len={metrics['eval/ep_len']:.0f}"
)
Video(video_path)

## Next steps

- Compare against vanilla PPO by rerunning with `--use-demos false` on the same seed.
- Tune `--bc-coef` (weaker/stronger persistent demo anchoring) or `--bc-pretrain-updates`.
- Full design notes: `examples/README.md` and the [Training with PPO](https://so101-nexus.com/docs/training/ppo) guide.